# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
from tqdm.notebook import tqdm
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, ParameterGrid, cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [18]:
class FeatureExtractor(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        df = X.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df['hour'] = df['timestamp'].dt.hour
        df['dayofweek'] = df['timestamp'].dt.dayofweek
        df = df.drop(columns=['timestamp'])
        return df

In [19]:
class MyOneHotEncoder(BaseEstimator, TransformerMixin):

    def __init__(self, target_col="dayofweek"):
        self.target_col = target_col

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        df = X.copy()

        target = df[self.target_col]
        features = df.drop(columns=[self.target_col])

        cat_cols = features.select_dtypes(include=['object', 'category', 'str']).columns.tolist()

        if cat_cols:
            encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
            encoded = encoder.fit_transform(features[cat_cols])
            features = pd.concat([
                features.drop(columns=cat_cols),
                pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_cols), index=features.index)
            ], axis=1)

        return features, target


In [20]:
class TrainValidationTest:

    def __init__(self, X, y):
        X_train_full, self.X_test, y_train_full, self.y_test = train_test_split(
            X, y, test_size=0.2, random_state=21, stratify=y
        )
        self.X_train, self.X_valid, self.y_train, self.y_valid = train_test_split(
            X_train_full, y_train_full, test_size=0.2, random_state=21, stratify=y_train_full
        )

    def get_splits(self):
        return (self.X_train, self.X_valid, self.X_test,
                self.y_train, self.y_valid, self.y_test)

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.877778
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.866667
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.907407
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [21]:
class ModelSelection:

    def __init__(self, grids, grid_dict):
        self.grids = grids
        self.grid_dict = grid_dict
        self.best_models = []
        self.best_name = None

    def choose(self, X_train, y_train, X_valid, y_valid):
        best_acc = 0
        self.best_models = []

        for idx, gs in enumerate(self.grids):
            name = self.grid_dict[idx]
            print(f"\nEstimator: {name}")

            param_list = list(ParameterGrid(gs.param_grid))

            best_cv_score = -np.inf
            best_params = None

            for params in tqdm(param_list):
                est = clone(gs.estimator)
                est.set_params(**params)
                cv_score = cross_val_score(
                    est, X_train, y_train,
                    cv=gs.cv, scoring=gs.scoring, n_jobs=gs.n_jobs
                ).mean()
                if cv_score > best_cv_score:
                    best_cv_score = cv_score
                    best_params = params

            best_estimator = clone(gs.estimator)
            best_estimator.set_params(**best_params)
            best_estimator.fit(X_train, y_train)

            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {best_cv_score:.3f}")

            valid_acc = accuracy_score(y_valid, best_estimator.predict(X_valid))
            print(f"Validation set accuracy score for best params: {valid_acc:.3f} ")

            self.best_models.append({
                "model": name, "params": best_params,
                "valid_score": valid_acc, "estimator": best_estimator
            })

            if valid_acc > best_acc:
                best_acc = valid_acc
                self.best_name = name
                self.best_estimator = best_estimator

        print(f"\nClassifier with best validation set accuracy: {self.best_name}")
        return self.best_name

    def best_results(self):
        rows = [{"model": m["model"], "params": m["params"], "valid_score": m["valid_score"]}
                for m in self.best_models]
        return pd.DataFrame(rows)

## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [22]:
class Finalize:

    def __init__(self, estimator):
        self.estimator = estimator
        self.accuracy = None

    def final_score(self, X_train, y_train, X_test, y_test):
        self.estimator.fit(X_train, y_train)
        y_pred = self.estimator.predict(X_test)
        self.accuracy = accuracy_score(y_test, y_pred)
        print(f"Accuracy of the final model is {self.accuracy}")
        return self.accuracy

    def save_model(self, path):
        joblib.dump(self.estimator, path)
        print(f"Model successfully saved to {path}")

## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [23]:
df = pd.read_csv('../data/checker_submits.csv')
print(df.shape)
df.head()

(1686, 4)


,uid,labname,numTrials,timestamp
0,user_4,project1,1,2020-04-17 05:19:02.744528
1,user_4,project1,2,2020-04-17 05:22:45.549397
2,user_4,project1,3,2020-04-17 05:34:24.422370
3,user_4,project1,4,2020-04-17 05:43:27.773992
4,user_4,project1,5,2020-04-17 05:46:32.275104


In [24]:
preprocessing = Pipeline([
    ('feature_extractor', FeatureExtractor()),
    ('onehot_encoder', MyOneHotEncoder('dayofweek')),
])

In [25]:
X, y  = preprocessing.fit_transform(df)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
X.head()

X shape: (1686, 43)
y shape: (1686,)


,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [26]:
splitter = TrainValidationTest(X, y)
X_train, X_valid, X_test, y_train, y_valid, y_test = splitter.get_splits()

print(f"X_train: {X_train.shape}, X_valid: {X_valid.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_valid: {y_valid.shape}, y_test: {y_test.shape}")

X_train: (1078, 43), X_valid: (270, 43), X_test: (338, 43)
y_train: (1078,), y_valid: (270,), y_test: (338,)


In [27]:
jobs = -1

svm_params = [{
    'kernel': ('linear', 'rbf', 'sigmoid'),
    'C': [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma': ['scale', 'auto'],
    'class_weight': ('balanced', None),
    'random_state': [21],
    'probability': [True],
}]

tree_params = [{
    'criterion': ('gini', 'entropy'),
    'max_depth': list(range(1, 30)),
    'class_weight': ('balanced', None),
    'random_state': [21],
}]

rf_params = [{
    'n_estimators': [10, 50, 100, 200],
    'criterion': ('gini', 'entropy'),
    'max_depth': list(range(1, 30)),
    'class_weight': ('balanced', None),
    'random_state': [21],
}]

In [28]:
gs_svm = GridSearchCV(
    estimator=SVC(),
    param_grid=svm_params,
    scoring='accuracy',
    cv=2,
    n_jobs=jobs,
)

gs_tree = GridSearchCV(
    estimator=DecisionTreeClassifier(),
    param_grid=tree_params,
    scoring='accuracy',
    cv=2,
    n_jobs=jobs,
)

gs_rf = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=rf_params,
    scoring='accuracy',
    cv=2,
    n_jobs=jobs,
)

In [29]:
grids = [gs_svm, gs_tree, gs_rf]
grid_dict = {0: "SVM", 1: "Decision Tree", 2: "Random Forest"}

ms = ModelSelection(grids, grid_dict)
best_name = ms.choose(X_train, y_train, X_valid, y_valid)


Estimator: SVM


  0%|          | 0/72 [00:00<?, ?it/s]

Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree


  0%|          | 0/116 [00:00<?, ?it/s]

Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 22, 'random_state': 21}
Best training accuracy: 0.809
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest


  0%|          | 0/464 [00:00<?, ?it/s]

Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest


In [33]:
ms.best_results()


,model,params,valid_score
0,SVM,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.877778
1,Decision Tree,"{'class_weight': 'balanced', 'criterion': 'gin...",0.866667
2,Random Forest,"{'class_weight': None, 'criterion': 'entropy',...",0.907407


In [31]:
final = Finalize(ms.best_estimator)
acc = final.final_score(X_train, y_train, X_test, y_test)

Accuracy of the final model is 0.9053254437869822


In [32]:
final.save_model('pipelines_best_model.sav')

Model successfully saved to pipelines_best_model.sav
